
# Inferencia

## Modelo
Lo primero que hay que hacer es aprender a cargar un modelo. Un modelo suele estar guardado en un repositorio HuggingFace junto con conjuntos de datos que se han usado para entrenarlo y otras cosas. El modelo está compuesto por varios ficheros:
1. Los pesos del modelo en un fichero formato .bin (Formato Torch antiguo) o formato .safetensors (Formato de HF). Podemos cargar los pesos del modelo con las funciones [AutoModelXX.from_pretrained()](https://huggingface.co/docs/transformers/model_doc/auto#generic-pretraining-classes). Cada vez que cargas un modelo se crea un directorio en la carpeta ~/.cache/huggingface/hub. Si no los borras de vez en cuando se te peta el sistema. Para borrarlos carga un entorno virtual de python donde tengas instalado el paquete __huggingface-hub__ y ejecuta el comando __hf cache scan__ para listar los modelos descargados y __hf cache delete__ para borrar algunos modelos.
   - AutoModel: carga un modelo base o solo el cuerpo de un modelo tuneado
   - AutoModelForCausalLM: carga un modelo con una capa de salida para generación de texto. si el modelo es un modelo base o tiene una capa de salida de otro tipo carga el modelo base y crea una capa de salida con pesos inicializados aleatoriamente.
   - AutoModelForSeq2SeqLM: para tareas de secuencia a secuencia con un encoder-decoder como T5 o BART
   - AutoModelForSequenceClassification: para tareas de clasificación de texto (Análisis de sentimiento, detección de spam, clasificación por categorías)
   - AutoModelForTokenClassification: Para tareas a nivel de palabra individual dentro de una frase (Reconocimiento de Entidades Nombradas como extraer nombres de personas, lugares o etiquetas gramaticales)
   - AutoModelForQuestionAnswering: Para Sistemas de respuestas. Recibe un contexto (ej. un artículo de Wikipedia) y una pregunta, y extrae del texto el fragmento exacto donde está la respuesta.
   - AutoModelForMaskedLM: Para predecir palabras ocultas (como el preentrenamiento original de BERT usando máscaras). Usado principalmente para entrenar modelos o rellenar huecos en textos.
   - AutoModelForMultipleChoice: Para problemas de opción múltiple (tipo examen de test), donde el modelo evalúa cuál de varias opciones de respuesta se adapta mejor a un contexto.
   - AutoModelForImageClassification: Para Clasificación de imágenes (ej. pasarle la foto de un animal y que te diga si es un "perro" o un "gato").
   - AutoModelForObjectDetection: Para Detección de objetos (Identifica qué objetos hay en una foto y te devuelve las coordenadas exactas de las cajas delimitadoras que los encierran).
   - AutoModelForSemanticSegmentation: Para Segmentación de imágenes. Etiqueta cada píxel de una foto de forma individual (esencial para sistemas de coches autónomos que separan la "carretera" de la "acera").
   - AutoModelForVideoClassification: Para clasificar fragmentos de vídeo enteros según la acción que ocurre en ellos (ej. "cocinar", "correr").
   - AutoModelForVision2Seq: Para modelos que reciben una imagen y generan texto (como los modelos VLM modernos tipo LLaVA, BLIP o las versiones vision de Llama). Sirve para que el modelo te describa detalladamente lo que ve en una foto o chatee contigo sobre ella.
   - AutoModelForCTC / AutoModelForSpeechSeq2Seq: Para Transcripción de audio a texto (Speech-to-Text).
   - AutoModelForAudioClassification: Para Clasificación de sonidos (ej. identificar qué instrumento musical suena o hacer reconocimiento de emociones por la voz).
Supón que ejecutamos el código
```
# use the same API for 3 different tasks
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
model = AutoModelForSequenceClassification.from_pretrained("meta-llama/Llama-2-7b-hf")
model = AutoModelForQuestionAnswering.from_pretrained("meta-llama/Llama-2-7b-hf")
```
El modelo llama-2-7b-hf es un modelo preentrenado pero no tuneado para hacer una tarea específica. La única instrucción que funcionaría sería la primera. Cargaría el modelo base junto con su capa original de generación de texto pero no sería capaz de ejecutar ninguna instrucción específica. Si lo que buscas es utilizar Llama-2 para chatear, que clasifique un texto o que responda una pregunta basándose en un contexto sin tener que entrenar tú las capas desde cero, no debes usar el modelo base. Debes usar las versiones -chat, las cuales han sido ajustadas por Meta mediante alineación humana (RLHF) y utilizar AutoModelForCausalLM. Las otras dos llamadas crearán una capa de salida con pesos aleatorios descartando la capa de salida original.
2. La arquitectura en un fichero llamado config.json. Podemos leer la arquitectura del modelo con la función AutoConfig.from_pretrained() sin necesidad de cargar los pesos inspeccionado la entrada **architectures**
   - Si pone BertModel es un modelo base
   - Si pone BertForCausalLM, su capa de salida es para generar texto.
   - Si pone BertForSequenceClassification, su capa de salida es un clasificador.
3. El preprocesador transforma los datos de entrada (texto, audio, imágenes o video) a grupos (batches) de tensores antes de introducirlos al modelo.
   - (tokenizador en el caso de LLM's se puede cargar con AutoTokenizer.from_pretrained()) y contiene los siguientes ficheros:
     - tokenizer.json (El estándar moderno): Es el archivo principal hoy en día. Contiene absolutamente todo lo necesario: el vocabulario completo, el algoritmo de tokenización (BPE, WordPiece, Unigram) y las reglas de preprocesamiento del texto.
     - tokenizer_config.json: Contiene los ajustes de configuración de la librería de Hugging Face para ese tokenizador. Por ejemplo, define qué tokens actúan como separadores ([SEP]), cuáles como inicio ([CLS]), o si debe convertir el texto a minúsculas automáticamente (do_lower_case: true)
     - special_tokens_map.json: Un pequeño archivo que le dice al sistema cómo mapear tokens especiales como el relleno ([PAD]) o palabras desconocidas ([UNK])
    - El audio (también el hablado) utiliza un Feature extractor para extraer características secuencias de formas de onda de audio y convertirlas en tensores.

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
#model_checkpoint = "bert-base-multilingual-cased"
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

## Tokenizador

El *tokenizer* devuelve un diccionario con tres items importantes:

- *input_ids*: son lo índices correspondientes a cada token de la frase.
- *attention_mask*: indica si un token debería ser *atendido* o no.
- *token_type_ids*: cuando hay más de una secuencia, identifica a qué secuencia (id) corresponde cada token

Fijamos el valor del parámetro return_tensors a pt para Pytorch o a tf para TensorFlow

In [3]:
encoded_input = tokenizer("Hello, my dog is cute", add_special_tokens=True, return_tensors="pt")

print(encoded_input)
result = tokenizer.decode(encoded_input["input_ids"][0])
print(result)

{'input_ids': tensor([[  101,  7592,  1010,  2026,  3899,  2003, 10140,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
[CLS] hello, my dog is cute [SEP]


Se puede ver que el tokenizer añadió dos tokens especiales: CLS y SEP (clasificado y separador) a la frase. No todos los modelos necesitan tokens especiales, pero si los necesitan, el tokenizer los añade automáticamente.

En el caso de tener varias entradas que queremos preprocesar, se las podemos pasar como una lista al tokenizer.

### Pad

Las frases no tienen siempre la misma longitud, por lo que no generarán tensores del mismo tamaño. Sin embargo, las entradas del modelo necesitan tener una forma uniforme. Para asegurarlo, se utiliza la técnica de padding que rellena los tensores más pequeños añadiendo un padding token especial a la codificación de las frases más cortas. Esto se hace automáticamente poniendo el parámetro padding a True.

Podemos observar que la primera y la última frase están rellenadas (padded) con 0's porque son más cortas.
Truncation

### Truncation

En el otro extremo, a veces tenemos en la entrada frases tan largas que exceden la dimensionalidad permitida por el modelo. En ese caso, vamos a acortar (truncate) la secuencia a una más corta. Para ello, vamos a poner el parámetro truncation a True para truncar la secuencia a la máxima longitud permitida por el modelo.


In [4]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt")
print(encoded_input)

{'input_ids': tensor([[  101,  2021,  2054,  2055,  2117,  6350,  1029,   102,     0,     0,
             0,     0,     0,     0],
        [  101,  2123,  1005,  1056,  2228,  2002,  4282,  2055,  2117,  6350,
          1010, 28315,  1012,   102],
        [  101,  2054,  2055,  5408, 14625,  1029,   102,     0,     0,     0,
             0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]])}


¡Ojo! Este modelo base tiene la cabeza aleatoria porque no ha sido entrenado para QA todavía.
Si quieres respuestas perfectas directamente sin entrenar, cambia el checkpoint por:  "mrm8488/bert-multi-cased-finetuned-xquadv1" (un BERT multilingüe ya entrenado para esta tarea)

In [3]:
messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
    {"role": "assistant", "content": "I don't know!"},
    {"role": "user", "content": "Are you sure?"},
]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted_chat)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(formatted_chat)


Dado que el modelo es un modelo base no incluye plantilla de chat como Qwen3 pero podemos crear nosotros de forma manual esa plantilla y especificarla a posteriori.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)
formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted_chat)

In [30]:
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

llm = Llama(
    model_path="/home/roberto/Applications/models/Qwen/Qwen3-1.7B-GGUF/Qwen3-1.7B-Q8_0.gguf",
    vocab_only=True, # Si solo queremos formatear el prompt lo ponemos a False
    verbose=False
)
# No me sirve formatter si uso vocab_only = True porque usa internamente
formatter = llm._chat_handlers["chat_template.default"]
plantilla_raw = llm.metadata.get("tokenizer.chat_template", None)
chat_template_str = plantilla_raw.decode('utf-8') if isinstance(plantilla_raw, bytes) else plantilla_raw
def obtener_token_metadata(llm_obj, nombre, por_defecto):
    token_id = llm_obj.metadata.get(f"tokenizer.ggml.{nombre}_token_id", None)
    if token_id is not None:
        return llm_obj.detokenize([int(token_id)]).decode('utf-8')
    return por_defecto
eos_real = obtener_token_metadata(llm, "eos", "<|im_end|>")
bos_real = obtener_token_metadata(llm, "bos", "<|endoftext|>")

# 3. Inicializamos de forma manual el formateador nativo de la suite de llama_cpp
# Pasamos la plantilla extraída y forzamos un tokenizador interno genérico
formatter = Jinja2ChatFormatter(
    template=chat_template_str,
    eos_token=eos_real,
    bos_token=bos_real,
    add_generation_prompt=True
)

# 4. Invocamos el formateador. En su versión nativa, la función 'format_chat' 
# procesa exclusivamente el string del prompt sin llamar a C++ y te devuelve el texto.
prompt_formateado = formatter(messages=messages).prompt
print(prompt_formateado)


<|im_start|>system
You are a friendly chatbot who always responds in the style of a pirate<|im_end|>
<|im_start|>user
How many helicopters can a human eat in one sitting?<|im_end|>
<|im_start|>assistant
I don't know!<|im_end|>
<|im_start|>user
Are you sure?<|im_end|>
<|im_start|>assistant



Ahora podemos pasar el texto formateado al llm.

In [48]:
llm = Llama(
    model_path="/home/roberto/Applications/models/Qwen/Qwen3-1.7B-GGUF/Qwen3-1.7B-Q8_0.gguf",
    vocab_only=False, # Si solo queremos formatear el prompt lo ponemos a False
    verbose=False
)
response = llm(
    prompt=prompt_formateado, 
    max_tokens=32
)
print(response["choices"][0]["text"])

<think>
Okay, the user asked how many helicopters a human can eat in one sitting. First, I need to respond in the pirate style. Pirates love a


Con la función create_chat_completion se hace todo en un solo paso

In [47]:
response = llm.create_chat_completion(
    messages=messages,
    max_tokens=32 # Lo ponemos a 0 si solo queremos procesar sin generar texto real
)
print(response["choices"][0]["message"])

{'role': 'assistant', 'content': "<think>\nOkay, the user asked how many helicopters a human can eat in one sitting, and I said I don't know. Then they asked if I was"}


La versión actual de transformers permite cargar checkpoints en formato gguf pero utiliza torch como motor lo cual no es muy óptimo. Llama-cpp-python dispone también del método from_pretrained para cargar modelos gguf directamente desde desde el hub de HuggingFace pero en este caso uiliza en motor nativo:
```
llm = Llama.from_pretrained(
   repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF", 
   filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"
)
```

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    "/home/roberto/Applications/models/Qwen/Qwen3-1.7B-GGUF", # ruta de la carpeta
    gguf_file="Qwen3-1.7B-Q8_0.gguf", # nombre del archivo
    device_map="auto", # Mueve automáticamente el modelo a tu GPU/CPU
    local_files_only=True # No buscar en el repositorio remoto de HuggingFace
)
print(model("¿Si tengo 15 manzanas y me como 3, cuántas me quedan?"))

Converting and de-quantizing GGUF tensors...:   0%|          | 0/310 [00:00<?, ?it/s]

## LM Studio
También podemos usar la aplicación LM Studio y pedirle que arranque un servidor

In [2]:
import requests

# 1. Definimos los datos del mensaje
payload = {"messages": 
           [
            {"role": "system", "content": "You are an assistant helping a student with their homework"},
            {"role": "assistant", "content": "Hi, how can I help you today?"},
            {"role": "user", "content": "I need help with my chemistry homework. Can you help me?"},
            {"role": "assistant", "content": "Of course! What specific topic or problem are you working on? Please provide more details so I can better assist you."},
            {"role": "user", "content": "Do you know how to formulate oxides?"},
           ]
          }

# 2. Hacemos la consulta directa a LM Studio
respuesta = requests.post("http://localhost:1234/v1/chat/completions", json=payload)

# 3. Extraemos el texto limpio del modelo
print(respuesta.json()["choices"][0]["message"]["content"])


<think>
Okay, the user asked if I know how to formulate oxides. Let me start by recalling what oxides are. Oxides are compounds made up of two elements, usually a metal and oxygen, or nonmetals and oxygen. The main thing is that they have an oxygen atom bonded to another element.

First, I should explain the basic concept. Maybe mention that oxides can be acidic, basic, or neutral depending on the element involved. For example, metals like sodium form basic oxides, while nonmetals like carbon form acidic ones. But wait, there's also metalloids and some elements with different properties.

I need to cover the types of oxides. There are metallic oxides, which are usually basic or amphoteric. Then there are nonmetallic oxides, which can be acidic or neutral. Also, maybe mention binary compounds where oxygen is combined with another element.

Wait, the user might not know the difference between different types. I should explain that oxides can be classified based on the element they're mad

Usando la librería openai

In [ ]:
import os
import openai

openai.api_base = "http://localhost:1234/v1" # point to the local server
openai.api_key = "" # no need for an API key

completion = openai.ChatCompletion.create(
  model="local-model", # this field is currently unused
  messages=[
    {"role": "system", "content": "You are an assistant helping a student with their homework"},
    {"role": "assistant", "content": "Hi, how can I help you today?"},
    {"role": "user", "content": "I need help with my chemistry homework. Can you help me?"},
    {"role": "assistant", "content": "Of course! What specific topic or problem are you working on? Please provide more details so I can better assist you."},
    {"role": "user", "content": "Do you know how to formulate oxides?"},
  ]
)

print(completion.choices[0].message)


## Análisis de sentimiento
BETO, es una versión de la arquitectura BERT preentrenada exclusivamente con un corpus masivo de texto en español. A diferencia de los modelos estrictamente binarios, este modelo fue entrenado con el corpus TASS (una competición de análisis de sentimiento en español). Gracias a esto, cuenta con tres neuronas de salida correspondientes a POSITIVO, NEGATIVO y NEUTRO. Mientras que los modelos pequeños pensados para frases cortas fallan o se sesgan rápidamente con párrafos grandes, la arquitectura basada en BETO soporta una longitud de contexto nativa de hasta 512 tokens.

In [ ]:
from transformers import pipeline

positivo="""
"Salvar al soldado Ryan" es una obra maestra cinematográfica que define el género bélico.
Steven Spielberg demuestra una vez más su destreza detrás de la cámara, capturando la crudeza y el heroísmo del conflicto como nunca antes se había visto.
La secuencia inicial del Día D es un tour de force de realización, con efectos de sonido y visuales que nos sumergen en el caos y el horror de la batalla.
Tom Hanks brinda una actuación espectacular como el capitán Miller, aportando una humanidad tangible a un contexto inhumano.
La trama es profundamente emotiva y plantea preguntas éticas que van más allá del simple relato de guerra.
Este film no es solo entretenimiento; es una lección de historia, humanidad y sacrificio.
"""

neutro="""
"Salvar al soldado Ryan" es una película que ha recibido elogios tanto de la crítica como del público, y es fácil ver por qué.
La dirección de Spielberg es competente y las actuaciones son sólidas en todo momento.
Sin embargo, la película tiene sus fallos.
Aunque la secuencia inicial del Día D es asombrosa en su realismo, el resto del film no mantiene ese nivel de intensidad.
Además, la trama es algo predecible y cae en algunos clichés del género.
Algunos personajes secundarios no están tan bien desarrollados como podrían haber estado, lo que resulta en una conexión emocional que no siempre se alcanza.
En resumen, es una película que vale la pena ver, pero quizás no sea para todos los gustos.
"""

negativo="""
A pesar de toda la aclamación que ha rodeado a "Salvar al soldado Ryan", me resultó decepcionante.
Si bien es cierto que la película tiene momentos de gran impacto visual, especialmente en su secuencia de apertura, estos no compensan sus numerosas deficiencias.
La narrativa es fragmentada y el ritmo es desigual; hay escenas que parecen arrastrarse interminablemente.
Aunque Tom Hanks hace un esfuerzo loable, su personaje carece de la profundidad necesaria para ser verdaderamente memorable.
Pero lo que más me molesta es cómo la película glorifica la guerra, en lugar de adoptar un enfoque más crítico o reflexivo.
En lugar de ser un retrato matizado del conflicto, se siente como un espectáculo de Hollywood que busca más emocionar que educar o provocar la reflexión.
"""

classifier = pipeline("sentiment-analysis", model="finiteautomata/beto-sentiment-analysis")

# Calcular el análisis de sentimientos para cada texto
resultado_positivo = classifier(positivo)[0]
resultado_neutro = classifier(neutro)[0]
resultado_negativo = classifier(negativo)[0]

# Imprimir los resultados
print("Análisis de sentimientos (positivo):", resultado_positivo)
print("Análisis de sentimientos (neutro):", resultado_neutro)
print("Análisis de sentimientos (negativo):", resultado_negativo)

La función [pipeline](https://huggingface.co/docs/transformers/v4.47.1/en/main_classes/pipelines#transformers.pipeline) permite utilizar cualquier modelo del [Hub](https://huggingface.co/models) the Huggingface para inferencia. Podemos usar la abstración pipeline para instanciar una de las siguientes tareas:
- 'audio-classification'
- 'automatic-speech-recognition'
- 'conversational'
- 'depth-estimation'
- 'document-question-answering' 
- 'feature-extraction' 
- 'fill-mask'
- 'image-classification'
- 'image-segmentation'
- 'image-to-text'
- 'mask-generation'
- 'ner'
- 'object-detection'
- 'question-answering'
- 'sentiment-analysis'
- 'summarization'
- 'table-question-answering'
- 'text-classification'
- 'text-generation'
- 'text2text-generation'
- 'token-classification'
- 'translation'
- 'video-classification' 
- 'visual-question-answering'
- 'vqa'
- 'zero-shot-audio-classification'
- 'zero-shot-classification'
- 'zero-shot-image-classification'
- 'zero-shot-object-detection'
- 'translation_XX_to_YY']

La función *pipeline()* puede recibir parámetros adicionales:

- Parámetros específicos de la configuración de un modelo determinado.
- Parámetros específicos de una tarea determinada. Por ejemplo en ASR existe la opción de especificar *return_timestamps=True* lo cual devolverá los *timestamps* en los que se pronunciaron las diferentes frases del discurso. Al ser disponibles de cada tarea su disponibilidad debe ser revisada en la documentación del pipeline específico para cada tarea. Por ejemplo, en la [documentación](https://huggingface.co/docs/transformers/v4.47.1/en/main_classes/pipelines#transformers.AutomaticSpeechRecognitionPipeline.__call__.return_timestamps) del *AutomaticSpeechRecognitionPipeline* se puede comprobar la disponibilidad del parámetro *return_timestamps*.
- *device=n*: Indicando el número del dispositivo en el que queremos localizar el modelo para inferencia.
- *batch_size=n*: El número de *batches* que se agruparán para inferencia. Se supone que tenemos que recibir un número de samples para inferencia mayor que *n*, y estos se le pasarán al modelo para inferencia en grupos de *n*. Por defecto, *batch_size=1* ya que utilizar *batches* mayores no tiene por qué ser necesariamente más rápido en todas las plataformas.

## Clasificación zero-shot

Esta tarea puede ser muy desafiante ya que necesitamos clasificar textos que no han sido etiquetados. Este es un escenario común en proyectos del mundo real porque anotar texto suele ser un proceso que consume mucho tiempo y requiere experiencia en el dominio. Para este caso de uso, el pipeline de clasificación zero-shot es muy útil: te permite especificar qué etiquetas usar para la clasificación, por lo que no tienes que depender de las etiquetas del modelo preentrenado. Ya has visto cómo el modelo puede clasificar una oración como positiva o negativa, pero también puede clasificar el texto usando cualquier otro conjunto de etiquetas que consideres.

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library in Natural Language Processing",
    candidate_labels=["education", "politics", "business"],
)

## Reconocimiento de entidades nombradas

El reconocimiento de entidades nombradas (NER) es una tarea donde el modelo tiene que encontrar qué partes del texto de entrada corresponden a entidades como personas, ubicaciones u organizaciones. Veamos un ejemplo:

In [ ]:
from transformers import pipeline

ner = pipeline("ner", grouped_entities=True)
ner("My name is Juan Pérez and I work at the University of Las Palmas of Gran Canaria in Spain.")

## Preguntas y respuestas

El pipeline de preguntas y respuestas responde preguntas utilizando información de un contexto dado:

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering")
question_answerer(
    question="What are María Pérez and Juan Pérez?",
    context="Juan Pérez and María Pérez work at the University of Las Palmas of Gran Canaria in Spain. They are colleagues. María Pérez is 32 years old and Juan Pérez is 45."
)

Una vez entrenado para ello el modelo bert dispone de dos vectores con las probabilidades de que cada token sea el comienzo y el final de la respuesta a una pregunta. 

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# Un ejemplo donde la respuesta es "1879" (un número que se fragmentará)
contexto = "Albert Einstein nacio en 1879 en Ulm."
pregunta = "¿En que ano nacio Einstein?"

# 1. Tokenizamos guardando el 'offset_mapping' (el mapa de caracteres)
inputs = tokenizer(
    pregunta,
    contexto,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# 2. Pasamos los inputs por el modelo sin calcular gradientes (modo evaluación)
with torch.no_grad():
    outputs = model(**inputs)

# 3. El modelo nos devuelve los LOGITS (puntuaciones) para cada token
start_logits = outputs.start_logits
end_logits = outputs.end_logits

# 4. Usamos argmax para que el modelo elija el token con la puntuación más alta
start_token = torch.argmax(start_logits).item()
end_token = torch.argmax(end_logits).item()

# Añadimos las etiquetas al diccionario final de entrenamiento
inputs["start_positions"] = start_token
inputs["end_positions"] = end_token

# Veamos qué tokens ha elegido el modelo
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"Tokens seleccionados como respuesta: {tokens[start_token : end_token + 1]}")
print(f"Etiqueta start_positions (Índice de token): {inputs['start_positions']}")
print(f"Etiqueta end_positions (Índice de token): {inputs['end_positions']}")

## Resúmenes

El resumen es la tarea de reducir un texto a un texto más corto mientras se mantienen todos (o la mayoría) de los aspectos importantes mencionados en el texto. Aquí hay un ejemplo:

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""
)

## T5
El modelo T5 (que veremos en el ejemplo), o Text-to-Text Transfer Transformer, fue presentado en un artículo titulado "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer" publicado por investigadores de Google en octubre de 2019. Este trabajo introdujo el modelo T5 junto con su enfoque innovador para manejar todas las tareas de procesamiento de lenguaje natural (NLP) como problemas de generación de texto, utilizando un formato de texto a texto.

T5ForConditionalGeneration es una clase en la biblioteca transformers de Hugging Face que proporciona una implementación del modelo T5 diseñada específicamente para tareas de generación de texto, como traducción de idiomas, resumen de texto, y generación de texto basada en prompts. La idea detrás de T5 es tratar todos los problemas de procesamiento de lenguaje natural como problemas de generación de texto. Por ejemplo, en lugar de tener una arquitectura específica para traducción y otra diferente para resumen, T5 puede ser entrenado para realizar ambas tareas (y muchas otras) simplemente cambiando el texto de entrada que se le proporciona. Esto se logra prefijando el texto de entrada con un indicador de la tarea a realizar, como "traducir del inglés al alemán" o "resumir", seguido por el texto a procesar.
### Traducción

In [7]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Cargamos el tokenizador y el modelo
model_name="t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

def do_task(text, task):

    # Preparamos la entrada
    input_text = f"{task}: {text}"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    # Generamos la salida
    outputs = model.generate(input_ids, max_length=300)
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return output_text

# Ejemplo de traducción del inglés al francés
text_en_to_es = "The Beatles were an English rock band formed in Liverpool in 1960, comprising John Lennon, Paul McCartney, George Harrison and Ringo Starr."
print("Inglés a Francés:", do_task(text_en_to_es, task="translate English to French"))
text_en_to_es = "They are regarded as the most influential band of all time and were integral to the development of 1960s counterculture and the recognition of popular music as an art form."
print("Inglés a Francés:", do_task(text_en_to_es, task="translate English to French"))

Castellano a Inglés: El Titanic, una obra maestra de la ingeniera naval de su tiempo, zarpó desde Southampton hacia Nueva York el 10 de abril de 1912, en su viaje inaugural. Promocionado como el transatlántico más grande y lujoso jamás construido y declarado "insumergible" por sus creadores, el barco


### Generación de resúmenes

In [ ]:
# Ejemplo de resumen
text = """
"The Beatles were an English rock band formed in Liverpool in 1960, comprising John Lennon, Paul McCartney, George Harrison and Ringo Starr. They are regarded as the most influential band of all time and were integral to the development of 1960s counterculture and the recognition of popular music as an art form.
"""
print("Resumen:", do_task(text, task="summarize"))



## SBERT

Usamos ahora el modelo Sentence Bert para calcular los embeddings de 3 textos y comprobar su similitud

In [1]:
texto1="""
El Titanic, una obra maestra de la ingeniería naval de su tiempo, zarpó desde Southampton hacia Nueva York el 10 de abril de 1912, en su viaje inaugural.
Promocionado como el transatlántico más grande y lujoso jamás construido y declarado "insumergible" por sus creadores, el barco era una representación palpable de los avances tecnológicos de la época.
A bordo iban pasajeros de diversas clases sociales, desde millonarios hasta emigrantes que buscaban un nuevo comienzo en América.
Su diseño incluía algunas de las tecnologías más avanzadas entonces disponibles, como compuertas estancas y un sistema de comunicación inalámbrica de última generación.

La tragedia se desató en la fría noche del 14 de abril de 1912, cuando el Titanic chocó contra un iceberg en el Atlántico Norte, a unas 375 millas al sur de Terranova.
Este evento fatal puso a prueba los límites de la ingeniería del barco y las prácticas de seguridad de la época.
A pesar de las avanzadas características de seguridad, el barco no contaba con suficientes botes salvavidas para todos sus pasajeros y tripulación.
Las fallas en las normas de seguridad y las malas decisiones tomadas durante y después del choque contribuyeron a la magnitud de la catástrofe.
En menos de tres horas, el Titanic se hundió en las gélidas aguas del Atlántico, llevando consigo a más de 1,500 de las aproximadamente 2,224 personas a bordo.

El hundimiento del Titanic no solo marcó una de las mayores tragedias marítimas en tiempos de paz sino que también tuvo un profundo impacto en la sociedad y la cultura.
El desastre provocó cambios significativos en las regulaciones de seguridad marítima, incluyendo la adopción de patrones más estrictos para los equipos de salvamento y las comunicaciones en los barcos.
Además, la historia del Titanic ha permeado la cultura popular, inspirando numerosos libros, películas y exposiciones que exploran el legado de este histórico naufragio.
La fascinación mundial con su historia continúa hasta hoy, sirviendo como un sombrío recordatorio de la fragilidad humana frente a las fuerzas de la naturaleza.
"""

# Prompt: "Vuelve a generar un texto de iguales características y temática, pero usa otro estilo (más novelesco y dramático) y otras palabras."

texto2="""
Era la noche del 14 de abril de 1912 cuando el majestuoso Titanic, un leviatán de acero que desafiaba los límites de la ingeniería moderna, se encontró con su destino en las heladas aguas del Atlántico Norte.
Como un palacio flotante, había zarpado de Southampton cargado de sueños y promesas, custodiando las vidas de aristócratas y aventureros, familias y solitarios, todos en busca de un futuro prometedor en las vastas tierras de América.
Bajo el manto estrellado, la noche se desgarró con el crujido del metal cuando el iceberg rasgó el casco del barco, y el insumergible comenzó su agonía en las profundidades gélidas.

En el caos que siguió, los destinos de más de dos mil almas se entrelazaron en una tragedia de proporciones épicas.
Las escenas a bordo eran de un desesperado ballet de supervivencia: mujeres y niños abrazados en lágrimas siendo bajados a los escasos botes salvavidas, mientras que los hombres aceptaban su sombrío destino con resignación o desesperación.
La orquesta, en un acto de desafío poético frente al inexorable avance del agua, tocaba melodías calmantes, una serenata para las almas que se desvanecían en la fría oscuridad del océano.

Al final, el Titanic se sumergió por completo, llevándose consigo historias no contadas y sueños no realizados.
Este desastre marítimo no solo se convirtió en un hito doloroso de la vulnerabilidad humana ante la naturaleza, sino también en un catalizador para reformar las leyes de seguridad marítima.
La tragedia del Titanic resonó en los corazones de la gente de todo el mundo, grabando su memoria en el tejido cultural de la humanidad, inspirando obras de arte, literatura y cine que buscan capturar la esencia de esa fatídica noche cuando el mar reclamó su trono frente a la arrogancia humana.
"""

# Prompt: "Genera un texto de tres párrafos sobre un hito cualquiera relacionado con el montañismo."

texto3="""
El montañismo ha sido testigo de numerosos hitos significativos, pero uno de los más destacados es la primera ascensión al Everest, la montaña más alta del mundo.
Este evento ocurrió el 29 de mayo de 1953, cuando el neozelandés Edmund Hillary y el sherpa nepalí Tenzing Norgay alcanzaron la cima del Everest.
Esta hazaña no solo fue un triunfo de la resistencia humana y el espíritu aventurero, sino que también marcó un punto de inflexión en la historia del montañismo, demostrando que era posible alcanzar los lugares más remotos y elevados de la Tierra.

Otro momento crucial en la historia del montañismo fue la primera ascensión invernal al K2, el segundo pico más alto del mundo y conocido por su dificultad extrema.
El 16 de enero de 2021, un equipo de alpinistas de Nepal consiguió llegar a la cima del K2 durante el invierno, una temporada en la que las condiciones son especialmente adversas debido a las temperaturas extremadamente bajas y los fuertes vientos.
Este logro no solo subrayó las capacidades extremas de los montañistas modernos, sino que también resaltó cómo el equipo y la tecnología en el montañismo han avanzado, permitiendo expediciones durante todo el año.

La evolución de la técnica y el equipo en el montañismo también ha permitido que más personas participen en este deporte, incluyendo a las mujeres y a personas de diversas nacionalidades.
Uno de los hitos más inspiradores en este ámbito fue la ascensión de Junko Tabei, quien se convirtió en la primera mujer en alcanzar la cima del Everest el 16 de mayo de 1975.
Este logro no solo desafió las percepciones tradicionales de género en el montañismo, sino que también abrió el camino para que muchas otras mujeres siguieran sus pasos en los años siguientes.
"""
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained model. Si lo quieres en castellano usa el primero
#model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
#model = SentenceTransformer(nomic-ai/nomic-embed-text-v2-moe') Hay GGUF
#model = SentenceTransformer('jinaai/jina-embeddings-v3')
model = SentenceTransformer('Alibaba-NLP/gte-multilingual-base', trust_remote_code=True) 

# Encode the texts y convierte los vectores a un tensor para poder usar pytorch_cos_sim
embeddings1 = model.encode(texto1, convert_to_tensor=True)
embeddings2 = model.encode(texto2, convert_to_tensor=True)
embeddings3 = model.encode(texto3, convert_to_tensor=True)

# Compute cosine similarities
cosine_scores12 = util.pytorch_cos_sim(embeddings1, embeddings2)
cosine_scores13 = util.pytorch_cos_sim(embeddings1, embeddings3)
cosine_scores23 = util.pytorch_cos_sim(embeddings2, embeddings3)

# Output the results
print("Cosine similarity between text1 and text2:", cosine_scores12.item())
print("Cosine similarity between text1 and text3:", cosine_scores13.item())
print("Cosine similarity between text2 and text3:", cosine_scores23.item())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/611M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine similarity between text1 and text2: 0.7367666959762573
Cosine similarity between text1 and text3: 0.42734280228614807
Cosine similarity between text2 and text3: 0.4050493836402893


## Voz a texto

In [49]:
from transformers import pipeline

t2s = pipeline("automatic-speech-recognition", model="openai/whisper-small")
t2s("data/voz.mp3")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cpu
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


{'text': ' Bueno, esto es solamente una prueba, a ver qué tal funciona el traductor de... bueno, no es un traductor realmente, es recoger el audio del ordenador y traducirlo a texto. Vamos a ver qué tal.'}

## Video
Podemos usar gradio para crear una interfaz web demostradora para el uso de un pipeline. 

In [ ]:
from transformers import pipeline
import gradio as gr

pipe = pipeline("image-classification", model="google/vit-base-patch16-224")

gr.Interface.from_pipeline(pipe).launch(share=True)